# Split donors / regions based on shapes from xenium explorer

In [1]:
import spatialdata as sd
import sparty as spt

# import spatialdata_io
# spatialdata_io.__version__
# '0.7.0'

/data/analysis/data_fierville/.envs/mf_isobrain/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Read xenium output 

In [ ]:
xenium_output = "DIR_PATH"

In [3]:
sdata = spt.io.load_xenium(xenium_output,  n_jobs = 15)
sdata

SpatialData object
├── Images
│     └── 'morphology_focus': DataTree[cyx] (4, 88331, 28440), (4, 44165, 14220), (4, 22082, 7110), (4, 11041, 3555), (4, 5520, 1777)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (88331, 28440), (44165, 14220), (22082, 7110), (11041, 3555), (5520, 1777)
│     └── 'nucleus_labels': DataTree[yx] (88331, 28440), (44165, 14220), (22082, 7110), (11041, 3555), (5520, 1777)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (33853, 1) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (34204, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (33853, 5101)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), nucleus_labels (Labels), transcripts (Points), cell_boundaries (Shapes), nucleus_boundaries (Shapes)

In [4]:
sdata['table'].uns['spatialdata_attrs']

{'region': 'cell_boundaries',
 'region_key': 'region',
 'instance_key': 'cell_id',
 'feature_key': 'feature_name'}

# Import shapes from xenium explorer

In [ ]:
file_sample_shapes = 'xenium_shapes.csv'

In [ ]:
gdf = spt.io.shapes_from_xe(
    file=file_sample_shapes,
    plot_fig = False,
    # return_gdf=False
)
gdf

,name,geometry
0,D655,"POLYGON ((968.46 2166.936, 2179.671 2003.013, ..."
1,D661,"POLYGON ((22197.707 13394.094, 22896.337 13483..."
2,D652,"POLYGON ((6720.018 43235.62, 7150.272 42882.04..."
3,D651,"POLYGON ((3467.867 54694.143, 3679.379 53094.5..."
4,D657,"POLYGON ((17771.5 68096.6, 18538.232 67224.112..."
5,D649,"POLYGON ((3741.384 80733.102, 4526.719 80448.6..."


In [24]:
# sdata.pl.render_shapes(
#     "cell_boundaries",
#     color='grey',
#     method='matplotlib',
#     ).pl.render_shapes(
#     "sample_shapes",
#     fill_alpha=0.5,
#     method='matplotlib',
#     outline_color='grey',
#     outline_width=1,
#     ).pl.show()

# Subset each shape and save as zarr 

In [ ]:
DIR = "OUT_DIR"

In [ ]:
for _, row in gdf.iterrows():
    name = row["name"]
    geometry = row["geometry"]
    print(name)

    cropped_sdata = sd.polygon_query(
        sdata,
        polygon=geometry,
        target_coordinate_system="global",
    )
    print(cropped_sdata)

    # scanpy prepro 
    # spt.pp.prepro_qc_scanpy(
    #     cropped_sdata, 
    #     min_counts = 10,
    #     min_genes = 5,
    #     pct_negative = 5.0)

    print(cropped_sdata['table'].n_obs)   
    cropped_sdata.write(f"{DIR}outs/sdata/sd_{name}.zarr")
    cropped_sdata['table'].write(f"{DIR}outs/adata/{name}.h5ad")



D655
SpatialData object
├── Images
│     └── 'morphology_focus': DataTree[cyx] (4, 6912, 8345), (4, 3456, 4173), (4, 1728, 2086), (4, 864, 1043), (4, 432, 521)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (6912, 8345), (3456, 4173), (1728, 2086), (864, 1043), (432, 521)
│     └── 'nucleus_labels': DataTree[yx] (6912, 8345), (3456, 4173), (1728, 2086), (864, 1043), (432, 521)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (5083, 1) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (5158, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (5083, 5101)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), nucleus_labels (Labels), transcripts (Points), cell_boundaries (Shapes), nucleus_boundaries (Shapes)
5083
D661
SpatialData object
├── Images
│     └── 'morphology_focus': DataTree[cyx] (4, 3591, 3783), (4, 1796, 